In [2]:
import numpy as np 
import pandas as pd 
import h5py
import re
import uproot
import glob 
import copy, sys, os
import matplotlib.pyplot as plt
from tqdm import tqdm 
# adding path to folder

from utils_analysis import ParticleCode, load_dataset
# Load your custom style 
plt.style.use('./cfg/my_custom_plot.mplstyle')

# use these lines on top of your matplotlib script
import matplotlib.ticker
class MyLocator(matplotlib.ticker.AutoMinorLocator):
    def __init__(self, n=4):
        super().__init__(n=n)
matplotlib.ticker.AutoMinorLocator = MyLocator        
 
# Now use matplotlib as usual.       
import matplotlib.pyplot as plt
plt.rcParams["xtick.minor.visible"] =  True
plt.rcParams["ytick.minor.visible"] =  True

sys.path.insert(0, '../tools/')
from caf_readers import CafReader
from mx2_matching import Mx2_DS_Match


In [3]:
# Load datasets 
n_files = 200
location = "nersc"
type="mr6"
df = load_dataset(n_files,location, type)
# Create a caf reader
caf_reader = CafReader(df)
pdg_tab = ParticleCode()

Reading  200  files gk
Location selected nersc
Openning MiniRun 6 CAFs


  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [07:35<00:00,  2.28s/it]


In [4]:
tpc_dist = 5
xbound = 63.931
ybound = 62.076
zbound = 64.3163

In [25]:
# Utility function to calculate cosL
def calculate_cosL(ev_data, ip):
    dz = ev_data['rec.mc.nu.prim.start_pos.z'][ip] - ev_data['rec.mc.nu.prim.end_pos.z'][ip]
    start_pos = np.array([
        ev_data['rec.mc.nu.prim.start_pos.x'][ip], 
        ev_data['rec.mc.nu.prim.start_pos.y'][ip],
        ev_data['rec.mc.nu.prim.start_pos.z'][ip]
    ])
    end_pos = np.array([
        ev_data['rec.mc.nu.prim.end_pos.x'][ip], 
        ev_data['rec.mc.nu.prim.end_pos.y'][ip],
        ev_data['rec.mc.nu.prim.end_pos.z'][ip]
    ])
    track_length = np.linalg.norm(start_pos - end_pos)
    return abs(dz / track_length)

# Step 1: Define the true neutrino signal
def true_neutrino_signal(ev_data, pdg_tab, xbound, ybound, zbound, tpc_dist): 
    signal = np.zeros_like(ev_data['rec.mc.nu.vtx.x'], dtype=bool)
    mask_t = (
        (abs(ev_data['rec.mc.nu.vtx.x']) < xbound - tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.x']) > tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.y']) < ybound - tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.z']) > tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.z']) < zbound - tpc_dist) &
        (ev_data['rec.mc.nu.targetPDG'] == pdg_tab.argon) &
        (ev_data['rec.mc.nu.iscc'] == 1) &
        (np.abs(ev_data['rec.mc.nu.pdg']) == pdg_tab.numu)  
    )
    signal[mask_t] = True
    
    for ix in range(ev_data['rec.mc.nu..length']):
        if mask_t[ix]: 
            n_particles = ev_data['rec.mc.nu.prim..length'][ix]
            n_pre = np.sum(ev_data['rec.mc.nu.prim..length'][:ix]) if ix > 0 else 0
            for ip in range(n_pre, n_pre + n_particles):
                pdg = ev_data['rec.mc.nu.prim.pdg'][ip]    
                if pdg == pdg_tab.muon: 
                    Elep = ev_data['rec.mc.nu.prim.p.E'][ip]
                    cosL = calculate_cosL(ev_data, ip)
                    if cosL < 0.9 and Elep < 1:
                        signal[ix] = False
    return signal

# Step 2: Define the reconstructed neutrino signal in FV
def reco_neutrino_signal(ev_data, ev, xbound, ybound, zbound, tpc_dist, mode):    
    mask_fv = (
        (abs(ev_data['rec.common.ixn.dlp.vtx.x']) < xbound - tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.x']) > tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.y']) < ybound - tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.z']) > tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.z']) < zbound - tpc_dist)
    )
    track_out_2x2 = track_selection_signal(ev_data)
    match_mx2 = track_match_mx2(ev_data,ev)
    if mode == 'FV': reco_vertices = mask_fv
    elif mode == 'exit_2x2': reco_vertices = (mask_fv)&(track_out_2x2)
    elif mode == 'match_mx2': reco_vertices = (mask_fv) & (match_mx2)
    else: raise ValueError("Invalid mode. Options: FV, exit_2x2, match_mx2") 

    return reco_vertices

# Step 2.5 select tracks that exit 2x2
def track_selection_signal(ev_data):
    #select vertices that are primary and exit 2x2
    z_end_track = ev_data['rec.common.ixn.dlp.part.dlp.end.z']
    is_reco_primary = ev_data['rec.common.ixn.dlp.part.dlp.primary']
    #array to store signal values
    track_out = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)

    for vtx in range(len(ev_data['rec.common.ixn.dlp.vtx.x'])):
        n_reco_tracks = ev_data['rec.common.ixn.dlp.part.dlp..length'][vtx]
        n_pre = np.sum(ev_data['rec.common.ixn.dlp.part.dlp..length'][:vtx]) if vtx > 0 else 0
        for ip in range(n_pre, n_pre + n_reco_tracks):
            if (is_reco_primary[ip] and z_end_track[ip]>=zbound):
                    track_out[vtx] = True
                    break 
    return track_out


def track_match_mx2(ev_data,ev):

    minerva_data = caf_reader.get_minerva_data(df.iloc[ev])
    is_reco_primary = ev_data['rec.common.ixn.dlp.part.dlp.primary']

    x_start_track  = ev_data['rec.common.ixn.dlp.part.dlp.start.x']
    y_start_track = ev_data['rec.common.ixn.dlp.part.dlp.start.y']
    z_start_track = ev_data['rec.common.ixn.dlp.part.dlp.start.z']

    x_end_track = ev_data['rec.common.ixn.dlp.part.dlp.end.x']
    y_end_track = ev_data['rec.common.ixn.dlp.part.dlp.end.y']
    z_end_track = ev_data['rec.common.ixn.dlp.part.dlp.end.z']

    track_Mx2 = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)

    for vtx in range(len(ev_data['rec.common.ixn.dlp.vtx.x'])):
        n_reco_tracks = ev_data['rec.common.ixn.dlp.part.dlp..length'][vtx]
        n_pre = np.sum(ev_data['rec.common.ixn.dlp.part.dlp..length'][:vtx]) if vtx > 0 else 0
        for ip in range(n_pre, n_pre + n_reco_tracks):
                if (is_reco_primary[ip] and z_end_track[ip] >= zbound):
                    muon_track = [[x_start_track[ip], y_start_track[ip],z_start_track[ip]],[x_end_track[ip],y_end_track[ip],z_end_track[ip]]]
                    reco_vtx = [ev_data['rec.common.ixn.dlp.vtx.x'][vtx],ev_data['rec.common.ixn.dlp.vtx.y'][vtx], ev_data['rec.common.ixn.dlp.vtx.z'][vtx]]
                    m_index,ds_dp,deltaX,deltaY,exit_minerva = Mx2_DS_Match(muon_track, minerva_data, reco_vtx, mc=True,verbose=False)
                    if (exit_minerva == True):
                        track_Mx2[vtx] == True
    return track_Mx2


# Step 3: Match reco vertices and true signal interactions
def matching(ev_data, vtx_index, signal, reco_vertices_signal):

    n_vtx = ev_data['rec.common.ixn.dlp.truth..length'][vtx_index]
    n_pre = np.sum(ev_data['rec.common.ixn.dlp.truth..length'][:vtx_index]) if vtx_index > 0 else 0 
    max_overlap = 0
    best_match_idx = None

    for ip in range(n_pre, n_pre + n_vtx):
        temp_overlap = ev_data['rec.common.ixn.dlp.truthOverlap'][ip]
        temp_tp_ixn = ev_data['rec.common.ixn.dlp.truth'][ip]
        if temp_overlap > max_overlap:
            max_overlap = temp_overlap
            best_match_idx = temp_tp_ixn

    if best_match_idx is not None:
        delta_x = np.abs(ev_data['rec.mc.nu.vtx.x'][best_match_idx] - ev_data['rec.common.ixn.dlp.vtx.x'][vtx_index])
        delta_y = np.abs(ev_data['rec.mc.nu.vtx.y'][best_match_idx] - ev_data['rec.common.ixn.dlp.vtx.y'][vtx_index])
        delta_z = np.abs(ev_data['rec.mc.nu.vtx.z'][best_match_idx] - ev_data['rec.common.ixn.dlp.vtx.z'][vtx_index])
        if signal[best_match_idx] and delta_x < 5 and delta_y < 5 and delta_z < 5:
            reco_vertices_signal[vtx_index] = True
            return True, best_match_idx, reco_vertices_signal
    return False, best_match_idx, reco_vertices_signal

#Step 4: get the type of interaction being reconstructed
def type_interaction(ev_data, scattering_counters, ScatteringMode, current_counters, CurrentMode, best_match_idx): 
    scattering_mode = ev_data['rec.mc.nu.mode'][best_match_idx]
    current_mode = ev_data['rec.mc.nu.iscc'][best_match_idx]
    if ev_data['rec.mc.nu.id'][best_match_idx] > 1E9:
        current_counters[CurrentMode['RockMuons']] += 1  # Increment rock_muons counter
    if scattering_mode in ScatteringMode.values() and current_mode ==1:
        scattering_counters[scattering_mode] += 1  # Increment the counter for the specific mode
    elif scattering_mode in ScatteringMode.values() and current_mode ==0:
        current_counters[CurrentMode['NC']] += 1  # Increment the counter for NC
    return scattering_counters, current_counters

# Helper function to calculate efficiency and purity
def efficiency_purity(count_match_FV, count_reco, num_neutrinos_signal_t):
    purity = count_match_FV / count_reco if count_reco > 0 else 0
    efficiency = count_match_FV / num_neutrinos_signal_t if num_neutrinos_signal_t > 0 else 0
    return efficiency, purity

# Plotting functions for interaction distributions
def plot_interactions(scattering_counters, current_counters, count_match_FV, ScatteringMode, CurrentMode):
    mode_labels = (
        [key for key, value in ScatteringMode.items() if value in scattering_counters] +
        [key for key, value in CurrentMode.items() if value in current_counters] +
        ['True Signal']
    )
    mode_counts = (
        [scattering_counters[ScatteringMode[label]] for label in ScatteringMode.keys()] +
        [current_counters[CurrentMode[label]] for label in CurrentMode.keys()] +
        [count_match_FV]
    )
    plt.figure(figsize=(7, 7))
    wedges, texts, autotexts = plt.pie(mode_counts, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
    plt.legend(wedges, mode_labels, title="Interaction Modes", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
    plt.title('Interaction Type of Reconstructed Vertices')
    plt.axis('equal')
    plt.show()

#plot type of scattering of the reco vertices that matched signal
def plot_signal_interactions(ScatteringMode, true_counters):
    labels = [label for label, code in ScatteringMode.items() if code in true_counters]
    sizes = [true_counters[ScatteringMode[label]] for label in labels]
    plt.figure(figsize=(7, 7))
    plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    plt.title('Scattering Mode of Vertices Matching Signal')
    plt.axis('equal')
    plt.show()


def display_summary(efficiency, purity, scattering_counters, current_counters, insignal_counters):

# Print and plot results    
    print("Interaction Summary:")
    total_interactions = sum(scattering_counters.values()) + sum(current_counters.values())
    for mode, count in {**scattering_counters, **current_counters}.items():
        percent = (count / total_interactions) * 100 if total_interactions > 0 else 0
        print(f"{mode}: {percent:.2f}%")
    
    print(f"\nEfficiency: {efficiency:.2%}")
    print(f"Purity: {purity:.2%}")




In [26]:
# Constants and initialization
ScatteringMode = {'CC kQE': 1, 'CC kDIS': 3, 'CC kRes': 4, 'CC kCoh': 5, 'CC kMEC': 10}
CurrentMode = {'NC': 0, 'RockMuons': 2}
scattering_counters = {mode: 0 for mode in ScatteringMode.values()}
current_counters = {mode: 0 for mode in CurrentMode.values()}
insignal_counters = {mode: 0 for mode in ScatteringMode.values()}

# Initialize counters and main loop
num_interactions, num_vertices, count_match_FV, num_match_signal = 0, 0, 0, 0
filtered_data = []

for ev in range(len(df)):
    ev_data = df.iloc[ev]
    #get true signal and reco vertices
    true_interactions = true_neutrino_signal(ev_data, pdg_tab, xbound, ybound, zbound, tpc_dist)
    reco_vertices = reco_neutrino_signal(ev_data, ev, xbound, ybound, zbound, tpc_dist, mode='match_mx2')
    match_vertices_signal = np.zeros_like(reco_vertices, dtype=bool)  
    #count number of vertices and interactions 
    num_vertices += np.sum(reco_vertices)
    num_interactions += np.sum(true_interactions)
    
    for vtx_index, value in enumerate(reco_vertices): 
        if value:  
            match, best_match_idx, match_vertices_signal = matching(ev_data, vtx_index, true_interactions, match_vertices_signal)
            if match: 
                count_match_FV += 1
                filtered_data.append(ev_data)
                #count type of interaction reconstructed in signal 
                insignal_counters[ev_data['rec.mc.nu.mode'][best_match_idx]] += 1    
            elif best_match_idx is not None:
                scattering_counters, current_counters = type_interaction(
                    ev_data, scattering_counters, ScatteringMode, current_counters, CurrentMode, best_match_idx
                )
    num_match_signal +=np.sum(match_vertices_signal)


print(f'There are {num_vertices} reconstructed vertices.')
print(f'There are {count_match_FV} reco vertices matching true interaction signal.')
print(f'There are {num_interactions} true interactions')
e, p = efficiency_purity(count_match_FV, num_vertices, num_interactions)
display_summary(e, p, scattering_counters, current_counters, insignal_counters)
plot_interactions(scattering_counters, current_counters, count_match_FV, ScatteringMode, CurrentMode)
plot_signal_interactions(ScatteringMode, insignal_counters)

There are 0 reconstructed vertices.
There are 0 reco vertices matching true interaction signal.
There are 2470 true interactions
Interaction Summary:
1: 0.00%
3: 0.00%
4: 0.00%
5: 0.00%
10: 0.00%
0: 0.00%
2: 0.00%

Efficiency: 0.00%
Purity: 0.00%


ValueError: cannot convert float NaN to integer

posx and posy should be finite values
posx and posy should be finite values


ValueError: need at least one array to concatenate

<Figure size 1400x1400 with 1 Axes>